In [ ]:
import os 

final_data_dir = "/home/elicer/dev/detectron2/final_data/train/images/"

num = os.listdir(final_data_dir)
print(len(num))

# train : 51872
# val : 12968개

51872


In [4]:
train_num = 51872
val_num = 12968
total_num = train_num + val_num

rat_train = round(train_num/total_num, 3) * 100
print(rat_train)

80.0


In [ ]:
### 샘플데이터 추출 ###

import os, random, shutil

def sample_data(img_dir, label_dir, out_img_dir, out_label_dir, sample_size):
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_label_dir, exist_ok=True)

    files = os.listdir(img_dir)
    random.seed(42)
    sampled = random.sample(files, sample_size)

    for f in sampled:
        # 이미지 복사
        shutil.copy(os.path.join(img_dir, f), os.path.join(out_img_dir, f))
        # 라벨 복사 (확장자 json으로 가정)
        base = os.path.splitext(f)[0]
        label_file = base + ".json"
        shutil.copy(os.path.join(label_dir, label_file), os.path.join(out_label_dir, label_file))

# 실행 예시
train_img_dir = "/home/elicer/dev/detectron2/final_data/train/images/"
train_label_dir = "/home/elicer/dev/detectron2/final_data/train/labels/"
out_train_img_dir = "/home/elicer/dev/ss/script/train_small_images"
out_train_label_dir = "/home/elicer/dev/ss/script/train_small_labels"

val_img_dir = "/home/elicer/dev/detectron2/final_data/val/images/"
val_label_dir ="/home/elicer/dev/detectron2/final_data/val/labels/"
out_val_img_dir = "/home/elicer/dev/ss/script/val_small_images"
out_val_label_dir = "/home/elicer/dev/ss/script/val_small_labels"

sample_data(train_img_dir, train_label_dir, out_train_img_dir, out_train_label_dir, 10000)
sample_data(val_img_dir, val_label_dir, out_val_img_dir, out_val_label_dir, 2000)

In [ ]:
### 폴더 내부 파일 새로운 폴더로 옮기기 ###

import os
import shutil

# 원본 폴더 경로
source_folder = "/home/elicer/dev/ss/script/val_small_images"

# JSON 파일을 옮길 대상 폴더 경로
target_folder = "/home/elicer/dev/ss/script/val_small_labels"

# 대상 폴더가 없으면 생성
os.makedirs(target_folder, exist_ok=True)

count = 0
# 폴더 안의 파일들 확인
for filename in os.listdir(source_folder):
    if filename.endswith(".json"):  # json 파일만 선택
        src_path = os.path.join(source_folder, filename)
        dst_path = os.path.join(target_folder, filename)
        shutil.move(src_path, dst_path)
        # print(f"Moved: {filename}")
        count += 1
print(f"JSON 파일 {count}개 이동 완료!")

JSON 파일 2000개 이동 완료!


In [ ]:
import json 


lst = list()

train_json_dir = "/home/elicer/dev/ss/data/coco_train_annotations.json"
with open(train_json_dir, 'r', encoding='utf-8') as f:
    data = json.load(f)

cats = data['categories']
for cat in cats:
    lst.append(cat['name'])


val_json_dir = "/home/elicer/dev/ss/data/coco_val_annotations.json"


with open(val_json_dir, 'r', encoding='utf-8') as f:
    data = json.load(f)

cats = data['categories']
for cat in cats:
    lst.append(cat['name'])


class_names = set(lst)

print(class_names)
print(len(class_names))

{'curb', 'sideWalk', 'vehicle', 'freespace', 'fense', 'stopLane', 'constructionGuide', 'eogVehicle', 'otherCar', 'roadMark', 'schoolBus', 'twoWheeler', 'crossWalk', 'rubberCone', 'rider', 'trafficSign', 'policeCar', 'safetyZone', 'whiteLane', 'trafficLight', 'truck', 'speedBump', 'ambulance', 'motorcycle', 'bicycle', 'trafficDrum', 'bus', 'yellowLane', 'pedestrian'}
29


In [ ]:
from detectron2.data.datasets import register_coco_instances
from detectron2.data import MetadataCatalog, DatasetCatalog

# 데이터셋 등록
train_json_dir = "/home/elicer/dev/ss/data/coco_train_annotations.json"
train_images_dir = "/home/elicer/dev/ss/data/train_small_images/"
val_json_dir = "/home/elicer/dev/ss/data/coco_val_annotations.json"
val_images_dir =  "/home/elicer/dev/ss/data/val_small_images/"

register_coco_instances("my_dataset_train", {}, train_json_dir, train_images_dir)
register_coco_instances("my_dataset_val", {}, val_json_dir,val_images_dir)

# 확인
metadata = MetadataCatalog.get("my_dataset_train")
dataset_dicts = DatasetCatalog.get("my_dataset_train")

In [4]:
from detectron2.data.datasets import register_coco_instances
from detectron2.engine import DefaultTrainer
from detectron2.config import get_cfg
import os

train_small_json = "/home/elicer/dev/ss/data/coco_train_annotations.json"
train_small_images = "/home/elicer/dev/ss/data/train_small_images/"
val_small_json = "/home/elicer/dev/ss/data/coco_val_annotations.json"
val_small_images = "/home/elicer/dev/ss/data/val_small_images/"


# Datasets (10k/2k)
register_coco_instances("my_dataset_train", {}, train_small_json, train_small_images)
register_coco_instances("my_dataset_val", {}, val_small_json, val_small_images)

cfg = get_cfg()
cfg.merge_from_file("/home/elicer/dev/ss/configs/COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")

cfg.DATASETS.TRAIN = ("my_dataset_train",)
cfg.DATASETS.TEST  = ("my_dataset_val",)
cfg.DATALOADER.NUM_WORKERS = 4

cfg.MODEL.WEIGHTS = "detectron2://COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x/137849600/model_final_f10217.pkl"

# Memory-safe choices
cfg.SOLVER.IMS_PER_BATCH = 2          # start safe; try 3 if memory allows
cfg.SOLVER.BASE_LR = 0.00025
cfg.SOLVER.MAX_ITER = 8000            # quick run for 10k
cfg.SOLVER.STEPS = []                 # no LR decay for quick experiment
cfg.SOLVER.AMP.ENABLED = True         # mixed precision for memory savings

# Input resizing (tune to your image sizes)
cfg.INPUT.MIN_SIZE_TRAIN = (640, 672, 704, 736, 768, 800)
cfg.INPUT.MAX_SIZE_TRAIN = 1333
cfg.INPUT.MIN_SIZE_TEST = 800
cfg.INPUT.MAX_SIZE_TEST = 1333
cfg.INPUT.RANDOM_FLIP = "horizontal"

cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 29    # match your categories

cfg.TEST.EVAL_PERIOD = 500
cfg.OUTPUT_DIR = "./output_10k_2k_safe"
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

[11/19 12:34:14 d2.engine.defaults]: Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res

/home/elicer/dev/detectron2/detectron2/engine/train_loop.py:474: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  grad_scaler = GradScaler()
Skip loading parameter 'roi_heads.box_predictor.cls_score.weight' to the model due to incompatible shapes: (81, 1024) in the checkpoint but (30, 1024) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.cls_score.bias' to the model due to incompatible shapes: (81,) in the checkpoint but (30,) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.bbox_pred.weight' to the model due to incompatible shapes: (320, 1024) in the checkpoint but (116, 1024) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.bbox_pred.bias' to the model due to incompatible shapes: (320,) in the checkpoint but (116

[11/19 12:34:20 d2.engine.train_loop]: Starting training from iteration 0


/home/elicer/dev/detectron2/detectron2/engine/train_loop.py:493: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=self.precision):
/home/elicer/miniconda3/envs/d2/lib/python3.10/site-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4314.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/home/elicer/miniconda3/envs/d2/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:182: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/d

[11/19 12:34:29 d2.utils.events]:  eta: 0:37:32  iter: 19  total_loss: 6.681  loss_cls: 3.444  loss_box_reg: 0.7036  loss_mask: 0.6933  loss_rpn_cls: 1.019  loss_rpn_loc: 0.8133    time: 0.2886  last_time: 0.1717  data_time: 0.0217  last_data_time: 0.0040   lr: 4.9953e-06  max_mem: 1773M


...

[11/19 13:10:27 d2.utils.events]:  eta: 0:00:10  iter: 7959  total_loss: 1.455  loss_cls: 0.2801  loss_box_reg: 0.3672  loss_mask: 0.2011  loss_rpn_cls: 0.04613  loss_rpn_loc: 0.5256    time: 0.2689  last_time: 0.4227  data_time: 0.0041  last_data_time: 0.0031   lr: 0.00025  max_mem: 2099M


/home/elicer/dev/detectron2/detectron2/engine/train_loop.py:493: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=self.precision):
/home/elicer/dev/detectron2/detectron2/engine/train_loop.py:493: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=self.precision):
/home/elicer/dev/detectron2/detectron2/engine/train_loop.py:493: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=self.precision):
/home/elicer/dev/detectron2/detectron2/engine/train_loop.py:493: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=self.precision):
/home/elicer/dev/detectron2/detectron2/engine/train_loop.py:493: FutureWarning: `torch.cuda.

[11/19 13:10:33 d2.utils.events]:  eta: 0:00:05  iter: 7979  total_loss: 1.518  loss_cls: 0.2648  loss_box_reg: 0.3688  loss_mask: 0.1939  loss_rpn_cls: 0.0478  loss_rpn_loc: 0.5509    time: 0.2689  last_time: 0.2498  data_time: 0.0042  last_data_time: 0.0047   lr: 0.00025  max_mem: 2099M


/home/elicer/dev/detectron2/detectron2/engine/train_loop.py:493: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=self.precision):
/home/elicer/dev/detectron2/detectron2/engine/train_loop.py:493: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=self.precision):
/home/elicer/dev/detectron2/detectron2/engine/train_loop.py:493: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=self.precision):
/home/elicer/dev/detectron2/detectron2/engine/train_loop.py:493: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=self.precision):
/home/elicer/dev/detectron2/detectron2/engine/train_loop.py:493: FutureWarning: `torch.cuda.

[11/19 13:10:39 d2.utils.events]:  eta: 0:00:00  iter: 7999  total_loss: 1.425  loss_cls: 0.2938  loss_box_reg: 0.3827  loss_mask: 0.1949  loss_rpn_cls: 0.05351  loss_rpn_loc: 0.4684    time: 0.2689  last_time: 0.4058  data_time: 0.0043  last_data_time: 0.0039   lr: 0.00025  max_mem: 2099M
[11/19 13:10:39 d2.engine.hooks]: Overall training speed: 7998 iterations in 0:35:51 (0.2689 s / it)
[11/19 13:10:39 d2.engine.hooks]: Total training time: 0:36:14 (0:00:23 on hooks)
[11/19 13:10:39 d2.data.datasets.coco]: Loaded 2000 images in COCO format from /home/elicer/dev/ss/data/coco_val_annotations.json
[11/19 13:10:39 d2.data.dataset_mapper]: [DatasetMapper] Augmentations used in inference: [ResizeShortestEdge(short_edge_length=(800, 800), max_size=1333, sample_style='choice')]
[11/19 13:10:39 d2.data.common]: Serializing the dataset using: <class 'detectron2.data.common._TorchSerializedList'>
[11/19 13:10:39 d2.data.common]: Serializing 2000 elements to byte tensors and concatenating them a

In [ ]:
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog
import cv2
import matplotlib.pyplot as plt

# 1. 설정 불러오기
cfg = get_cfg()
cfg.merge_from_file("path/to/config.yaml")  # 학습에 사용한 config 파일 경로
cfg.MODEL.WEIGHTS = "output_10k_2k_safe/model_final.pth"
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5  # 신뢰도 threshold
cfg.MODEL.DEVICE = "cuda"  # GPU 사용 (없으면 "cpu")

# 2. Predictor 생성
predictor = DefaultPredictor(cfg)

# 3. 이미지 불러오기
image = cv2.imread("path/to/test_image.jpg")

# 4. 추론 실행
outputs = predictor(image)

# 5. 결과 시각화
v = Visualizer(image[:, :, ::-1], MetadataCatalog.get(cfg.DATASETS.TRAIN[0]), scale=1.2)
out = v.draw_instance_predictions(outputs["instances"].to("cpu"))
result_image = out.get_image()

# 6. 결과 출력
plt.figure(figsize=(12, 8))
plt.imshow(result_image)
plt.axis("off")
plt.show()